# RAG Anything — Search, Download, Ingestion Showcase

This notebook demonstrates a practical search workflow over local files and URL manifests, optional local download of URL-based assets, batch ingestion into AIFluent, and a thesaurus-style summary of file sizes, file lists, and document groups.


## Scope

The showcase is designed to cover these source families with **3 examples per type** when data is available locally or via URL manifests:

- PDF: `.pdf`
- MS Word: `.docx`
- MS Excel: `.xlsx`
- MS PowerPoint: `.pptx`
- Images: `.jpg`, `.png`, `.gif`
- Video: `.mp4`
- Audio: `.mp3`
- Markdown text: `.md`
- Code / semi-structured text: `.txt`, `.json`, `.yaml`
- Text chunks / fragments in `.txt` and URL manifests pointing to multimedia content

The notebook reports what is currently present, what is missing, and which subsets are ready for ingestion.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.notebook_utils import (
    ensure_project_root_on_path,
    ensure_dir,
    RAW_DATA,
    OUTPUT_DIR,
)

ensure_project_root_on_path()
ensure_dir(RAW_DATA)
ensure_dir(OUTPUT_DIR)

from src.document_inventory import collect_inventory_from_root, summarize_inventory, showcase_coverage
from src.search_ingestion import build_request, discover_candidates, execute_search_ingestion_sync

DATA_ROOT = RAW_DATA
URL_MANIFEST_DIR = ensure_dir(RAW_DATA / "url_manifests")
print(f"Data root: {DATA_ROOT}")
print(f"URL manifest dir: {URL_MANIFEST_DIR}")


## Optional URL Manifest Templates

Use these manifests when you want the search module to download remote assets locally before ingestion. One URL per line in `.txt`, or use `.json` / `.csv` manifests supported by the search module.


In [ ]:
manifest_examples = {
    "images.txt": [
        "https://example.com/path/sample-image-01.jpg",
        "https://example.com/path/sample-image-02.png",
        "https://example.com/path/sample-image-03.gif",
    ],
    "videos.txt": [
        "https://media.example.com/path/sample-video-01.mp4",
        "https://media.example.com/path/sample-video-02.mp4",
        "https://media.example.com/path/sample-video-03.mp4",
    ],
    "audio.txt": [
        "https://media.example.com/path/sample-audio-01.mp3",
        "https://media.example.com/path/sample-audio-02.mp3",
        "https://media.example.com/path/sample-audio-03.mp3",
    ],
}

for filename, urls in manifest_examples.items():
    path = URL_MANIFEST_DIR / filename
    if not path.exists():
        path.write_text("
".join(urls) + "
")

print("Manifest templates available:")
for path in sorted(URL_MANIFEST_DIR.glob('*')):
    print(' -', path)


## Build The Local Inventory And Thesaurus

This cell scans `data/raw/`, computes per-file size, classifies files into document groups, and creates a thesaurus structure keyed by document type.


In [ ]:
inventory_entries = collect_inventory_from_root(DATA_ROOT, recursive=True)
inventory_summary = summarize_inventory(inventory_entries)
coverage = showcase_coverage(inventory_entries)

print(json.dumps({
    "total_files": inventory_summary["total_files"],
    "total_size_bytes": inventory_summary["total_size_bytes"],
    "groups": inventory_summary["groups"],
}, indent=2))


In [ ]:
pd.DataFrame(inventory_summary["files"]).head(50) if inventory_summary["files"] else pd.DataFrame(columns=["path", "name", "extension", "group", "size_bytes"])


In [ ]:
coverage_rows = []
for label, info in coverage.items():
    coverage_rows.append({
        "label": label,
        "required": info["required"],
        "targets": ", ".join(info["targets"]),
        "available_total": sum(info["available"].values()),
        "available_detail": json.dumps(info["available"]),
        "ready": info["meets_requirement"],
    })

pd.DataFrame(coverage_rows)


## Thesaurus View

The thesaurus is a dictionary keyed by normalized document group. Each group lists the matching files with size and extension metadata.


In [ ]:
thesaurus = inventory_summary["thesaurus"]
print(json.dumps(thesaurus, indent=2)[:12000])


## Search Examples Over Local Data

The search module supports: keywords, document types, extensions, input location, URL subdomain filtering, and output location. These examples are dry-run searches against local files and URL manifests.


In [ ]:
local_search_requests = {
    "pdfs": build_request(
        keywords="report,whitepaper,guide",
        document_types="pdf",
        input_location=str(DATA_ROOT),
        dry_run=True,
    ),
    "office_docs": build_request(
        keywords="deck,slides,budget,workbook,proposal",
        document_types="word,excel,powerpoint",
        input_location=str(DATA_ROOT),
        dry_run=True,
    ),
    "media_urls": build_request(
        keywords="sample",
        extensions=".jpg,.png,.gif,.mp4,.mp3",
        input_location=str(URL_MANIFEST_DIR / "images.txt"),
        url_subdomain_pattern="*.example.com",
        dry_run=True,
    ),
}

for name, request in local_search_requests.items():
    candidates = discover_candidates(request)
    print(f"
=== {name} ===")
    print(f"Matched candidates: {len(candidates)}")
    for candidate in candidates[:10]:
        print(f" - {candidate.document_type:12} {candidate.reference}")


## Optional Download Then Ingest

This section shows a full search-and-ingest request. Keep `dry_run=True` until your local files, manifests, parser dependencies, and model environment are ready.

Set `dry_run=False` to actually download URL candidates (if any) and ingest matching files through AIFluent.


In [ ]:
showcase_request = build_request(
    keywords="report,guide,sample",
    document_types="pdf,word,excel,powerpoint,image,video,audio,markdown,code",
    input_location=str(DATA_ROOT),
    output_location=str(OUTPUT_DIR / "showcase"),
    parse_method="auto",
    workers=1,
    recursive=True,
    dry_run=True,
)

showcase_result = execute_search_ingestion_sync(showcase_request)
print(json.dumps(showcase_result.to_dict(), indent=2)[:12000])


## Read Three Files Per Type Where Available

This helper reads up to 3 items per document group from the local thesaurus. For binary formats it reports metadata only. For text-like formats it previews the first 400 characters.


In [ ]:
TEXTLIKE_EXTENSIONS = {".md", ".txt", ".json", ".yaml", ".yml"}

preview_rows = []
for group, items in thesaurus.items():
    for item in items[:3]:
        path = Path(item["path"])
        preview = "<binary>"
        if path.suffix.lower() in TEXTLIKE_EXTENSIONS:
            try:
                preview = path.read_text(encoding="utf-8")[:400]
            except Exception as exc:
                preview = f"<read failed: {exc}>"
        preview_rows.append({
            "group": group,
            "name": item["name"],
            "extension": item["extension"],
            "size_bytes": item["size_bytes"],
            "preview": preview,
        })

pd.DataFrame(preview_rows)


## Notes

- Parsers for `docx`, `xlsx`, `pptx`, images, audio, and video depend on your installed parser stack and may require additional system dependencies.
- URL-manifest based download uses the same search module as the CLI and web UI.
- For a full run, make sure `.env` is configured, model adapters are available, and `dry_run=False` is set on the search-ingest request.
